In [1]:

# CLASSIFICATION: SI > 8

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)


# 1. Загрузка данных


sheet_id = "1q-nbWuFrfrIBMXmZfNW78N3bx5v60Vb9"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

df = pd.read_csv(url)

print("Размер датасета:", df.shape)
display(df.head())


# 2. Базовая предобработка


if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

print("Размер после удаления служебного столбца:", df.shape)

print("\nКоличество пропусков:")
print(df.isna().sum().sum())

df = df.fillna(df.median(numeric_only=True))


# 3. Создание целевой переменной


threshold = 8
print(f"\nПорог для классификации SI: {threshold}")

# 1 -> SI > 8
# 0 -> SI <= 8
df["SI_gt8_class"] = (df["SI"] > threshold).astype(int)

print("\nРаспределение классов:")
print(df["SI_gt8_class"].value_counts())
print(df["SI_gt8_class"].value_counts(normalize=True))


# 4. Формирование признаков


# Убираю SI, IC50 и CC50, чтобы не было утечки
drop_cols = ["SI", "IC50, mM", "CC50, mM", "SI_gt8_class"]

X = df.drop(columns=drop_cols, errors="ignore")
y = df["SI_gt8_class"]

print("\nРазмер X:", X.shape)
print("Размер y:", y.shape)


# 5. Train / Test split


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)


# 6. Модель 1 — Logistic Regression


logreg_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000, random_state=42))
])

logreg_model.fit(X_train, y_train)
y_pred_logreg = logreg_model.predict(X_test)

print("=" * 60)
print("Logistic Regression")
print("=" * 60)
print("Accuracy:", round(accuracy_score(y_test, y_pred_logreg), 4))
print("F1-score:", round(f1_score(y_test, y_pred_logreg), 4))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred_logreg))
print("\nClassification report:")
print(classification_report(y_test, y_pred_logreg))


# 7. Модель 2 — Random Forest


rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print("=" * 60)
print("Random Forest")
print("=" * 60)
print("Accuracy:", round(accuracy_score(y_test, y_pred_rf), 4))
print("F1-score:", round(f1_score(y_test, y_pred_rf), 4))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred_rf))
print("\nClassification report:")
print(classification_report(y_test, y_pred_rf))


# 8. Сводная таблица результатов


results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_logreg),
        accuracy_score(y_test, y_pred_rf)
    ],
    "F1-score": [
        f1_score(y_test, y_pred_logreg),
        f1_score(y_test, y_pred_rf)
    ]
})

print("=" * 60)
print("Сравнение моделей")
print("=" * 60)
display(results.sort_values(by="F1-score", ascending=False).reset_index(drop=True))

best_model_name = results.sort_values(by="F1-score", ascending=False).iloc[0]["Model"]
print(f"Лучшая модель по F1-score: {best_model_name}")


# 9. Важность признаков для Random Forest


feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf_model.feature_importances_
}).sort_values(by="importance", ascending=False)

print("\nТоп-15 важных признаков:")
display(feature_importance.head(15))




Размер датасета: (1001, 214)


,Unnamed: 0,"IC50, mM","CC50, mM",SI,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,0,6.239374,175.482382,28.125000,5.094096,5.094096,0.387225,0.387225,0.417362,42.928571,...,0,0,0,0,0,0,0,0,3,0
1,1,0.771831,5.402819,7.000000,3.961417,3.961417,0.533868,0.533868,0.462473,45.214286,...,0,0,0,0,0,0,0,0,3,0
2,2,223.808778,161.142320,0.720000,2.627117,2.627117,0.543231,0.543231,0.260923,42.187500,...,0,0,0,0,0,0,0,0,3,0
3,3,1.705624,107.855654,63.235294,5.097360,5.097360,0.390603,0.390603,0.377846,41.862069,...,0,0,0,0,0,0,0,0,4,0
4,4,107.131532,139.270991,1.300000,5.150510,5.150510,0.270476,0.270476,0.429038,36.514286,...,0,0,0,0,0,0,0,0,0,0


Размер после удаления служебного столбца: (1001, 213)

Количество пропусков:
36

Порог для классификации SI: 8

Распределение классов:
SI_gt8_class
0    644
1    357
Name: count, dtype: int64
SI_gt8_class
0    0.643357
1    0.356643
Name: proportion, dtype: float64

Размер X: (1001, 210)
Размер y: (1001,)

Train shape: (800, 210)
Test shape: (201, 210)
Logistic Regression
Accuracy: 0.6716
F1-score: 0.5286

Confusion matrix:
[[98 31]
 [35 37]]

Classification report:
              precision    recall  f1-score   support

           0       0.74      0.76      0.75       129
           1       0.54      0.51      0.53        72

    accuracy                           0.67       201
   macro avg       0.64      0.64      0.64       201
weighted avg       0.67      0.67      0.67       201

Random Forest
Accuracy: 0.7313
F1-score: 0.5645

Confusion matrix:
[[112  17]
 [ 37  35]]

Classification report:
              precision    recall  f1-score   support

           0       0.75      0.87

,Model,Accuracy,F1-score
0,Random Forest,0.731343,0.564516
1,Logistic Regression,0.671642,0.528571


Лучшая модель по F1-score: Random Forest

Топ-15 важных признаков:


,feature,importance
21,BCUT2D_CHGLO,0.020050
103,VSA_EState8,0.018393
99,VSA_EState4,0.017983
68,SMR_VSA7,0.017710
1,MaxEStateIndex,0.016300
93,EState_VSA8,0.016232
22,BCUT2D_LOGPHI,0.016036
0,MaxAbsEStateIndex,0.015707
105,FractionCSP3,0.015057
5,SPS,0.014284


## Выводы

В этой части я решала задачу бинарной классификации: определяла, превышает ли значение **SI** фиксированный порог **8**.

Сначала я загрузила датасет и выполнила базовую предобработку. Исходный размер таблицы составил **1001 строку и 214 столбцов**. После удаления служебного столбца размер данных стал **1001 × 213**. Затем я проверила данные на пропуски и обнаружила **36 пропущенных значений**, после чего заполнила их медианными значениями.

Далее я сформировала новую бинарную целевую переменную **SI_gt8_class**:
- класс **1** — если **SI > 8**
- класс **0** — если **SI ≤ 8**

Распределение классов оказалось несбалансированным:
- класс **0** — **644 объекта** (**64.34%**)
- класс **1** — **357 объектов** (**35.66%**)

После этого я сформировала матрицу признаков и целевую переменную. При этом я исключила из признаков **SI**, а также **IC50** и **CC50**, так как **SI** рассчитывается на их основе. Это было необходимо, чтобы избежать утечки данных и получить более корректную оценку качества моделей.

После удаления целевых и связанных столбцов размер матрицы признаков составил **(1001, 210)**, а размер целевой переменной — **(1001,)**.

Затем я разделила данные на обучающую и тестовую выборки:
- **train:** **800 объектов**, **210 признаков**
- **test:** **201 объект**, **210 признаков**

После подготовки данных я обучила две модели:
- **Logistic Regression**
- **Random Forest**

Далее я сравнила их по основным метрикам качества: **Accuracy**, **F1-score**, **матрице ошибок** и **classification report**.

Также я дополнительно проанализировала важность признаков для модели **Random Forest**, чтобы определить, какие дескрипторы химических соединений сильнее всего влияют на итоговое предсказание.

### Logistic Regression

Модель **Logistic Regression** показала следующие результаты:
- **Accuracy:** **0.6716**
- **F1-score:** **0.5286**

Матрица ошибок:
```text
[[98 31]
 [35 37]]

Из classification report:

для класса 0:
precision = 0.74
recall = 0.76
f1-score = 0.75
для класса 1:
precision = 0.54
recall = 0.51
f1-score = 0.53

Итоговая accuracy по тестовой выборке составила 0.67.

Random Forest

Модель Random Forest показала следующие результаты:

Accuracy: 0.7313
F1-score: 0.5645

Матрица ошибок:

[[112 17]
 [ 37 35]]

Из classification report:

для класса 0:
precision = 0.75
recall = 0.87
f1-score = 0.81
для класса 1:
precision = 0.67
recall = 0.49
f1-score = 0.56

Итоговая accuracy по тестовой выборке 0.73.

Сравнение моделей

По итогам сравнения моделей я получила  результаты:

Random Forest
Accuracy: 0.731343
F1-score: 0.564516
Logistic Regression
Accuracy: 0.671642
F1-score: 0.528571

Лучшей моделью по F1-score стала Random Forest. Также эта модель показала и более высокую Accuracy, поэтому на текущем этапе именно её я считаю более удачным решением для задачи классификации SI > 8.

Наиболее важные признаки

Дополнительно я проанализировала важность признаков, рассчитанную моделью Random Forest. Наиболее значимыми оказались следующие дескрипторы:

BCUT2D_CHGLO — 0.020050
VSA_EState8 — 0.018393
VSA_EState4 — 0.017983
SMR_VSA7 — 0.017710
MaxEStateIndex — 0.016300
EState_VSA8 — 0.016232
BCUT2D_LOGPHI — 0.016036
MaxAbsEStateIndex — 0.015707
FractionCSP3 — 0.015057
Итоговый вывод

В результате я получила рабочее решение для задачи SI > 8. По сравнению с задачей классификации по медиане эта постановка оказалась сложнее, так как распределение классов здесь уже несбалансированное: доля класса 1 составила только 35.66%.

На текущем этапе лучшей моделью я считаю Random Forest, поскольку именно она показала лучший результат по метрике F1-score = 0.5645 и более высокую Accuracy = 0.7313.

При этом видно, что даже лучшая модель заметно лучше распознаёт класс 0, чем класс 1: recall для положительного класса составил только 0.49. Это означает, что модель пока недостаточно хорошо выявляет все соединения с SI > 8, и в дальнейшем качество можно улучшить с помощью подбора гиперпараметров, балансировки классов и более глубокого отбора признаков.